# 월별 환승 편의성(MonthlyTransferAccessibility) → DuckDB 적재

공공데이터포털 `TransferAccessibility/getMonthlyTransferAccessibility`.

⚠️ **연별 API 없어서 월별 사용**. 데이터 크기 우려로 **1개월(2025-10) 임시 수집**.
→ 1개월 결과 보고 12개월 풀/4계절 샘플 등 결정.

- `opr_ym` = **202510** (2025년 10월, 임시 테스트용)
- `ctpv_cd` = 11 (서울 고정), `sgg_cd` = 서울 25개 구
- **환승시간(trnf_hr)**: 초 단위 총합 → `trnf_hr / pasg_cnt` = 평균 환승시간
- **노인 보행 프로젝트 활용**: 환승시간 긴 정류소 × 노인 이용량 → 접근성 핫스팟
- PK: (opr_ym, sttn_id, inoutf_type_cd, trnf_type_cd, tzon)

나중에 월 추가하려면 셀 4의 `YEAR_MONTHS`에 원하는 YYYYMM 추가 후 다시 실행 (재개 지원됨).

In [1]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저 (예외 시에도 안전)."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

SERVICE_KEY 로드됨 (길이 88)
DB: data/seoul.duckdb (헬퍼 준비됨)


In [2]:
# ============================================================
# 셀 2 - Pydantic 모델
# ============================================================
from typing import Generic, TypeVar, Optional
from pydantic import BaseModel, field_validator

T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    item: list[T] = []

    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":
            return []
        if isinstance(v, dict):
            return [v]
        return v


class Body(BaseModel, Generic[T]):
    items: Items[T] = Items[T]()
    dataType: str | None = None
    pageNo: int
    numOfRows: int
    totalCount: int


class Response_(BaseModel, Generic[T]):
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    Response: Response_[T]


class TransferAccessibilityItem(BaseModel):
    """월별 환승 편의성 한 건 (정류소 × 시간대 × 환승유형별 통계).

    PK: (opr_ym, sttn_id, inoutf_type_cd, trnf_type_cd, tzon)

    필드 의미:
      - trnf_hr: 총 환승시간 (초 단위 합계)
        → 평균 환승시간 = trnf_hr / pasg_cnt
      - trnf_type_cd: TB=도시철도→버스, BT=버스→도시철도,
                      BB=버스→버스, TT=도시철도→도시철도
      - inoutf_type_cd: I=유입(들어옴), O=유출(나감)
      - trfc_mns_se_cd: T=전철, B=버스 (환승 기준점 교통수단)
      - pasg_cnt: 환승 건수 (중복 포함)
      - pasg_nope: 환승 이용자 수
    """
    # PK 필수
    opr_ym: str
    sttn_id: str
    inoutf_type_cd: str
    trnf_type_cd: str
    tzon: str

    # 지역 코드 (region JOIN용)
    ctpv_cd: Optional[str] = None
    sgg_cd: Optional[str] = None
    emd_cd: Optional[str] = None
    trfc_mns_se_cd: Optional[str] = None

    # 지표
    trnf_hr: int = 0       # 총 환승시간(초)
    pasg_cnt: int = 0      # 환승 건수
    pasg_nope: int = 0     # 환승 이용자 수

    # 응답에만 있고 저장 안 함 (정규화 — region/코드 해석표에서 JOIN)
    ctpv_nm: Optional[str] = None
    sgg_nm: Optional[str] = None
    emd_nm: Optional[str] = None
    inoutf_type_nm: Optional[str] = None
    trnf_type_nm: Optional[str] = None


print("모델 로드 완료")

모델 로드 완료


In [3]:
# ============================================================
# 셀 3 - fetch 함수 (NO_DATA_FOUND 에러 처리 포함)
# ============================================================
TRANSFER_URL = "https://apis.data.go.kr/1613000/TransferAccessibility/getMonthlyTransferAccessibility"


def fetch_all_pages(
    url: str,
    item_model: type[T],
    extra_params: dict | None = None,
    num_rows: int = 1000,
) -> list[T]:
    """공공데이터 표준 응답 API 전체 페이지 수집. NO_DATA_FOUND는 빈 결과로 취급."""
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }
    all_items: list[T] = []
    page = 1
    while True:
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=60)
        res.raise_for_status()
        payload = res.json()

        if "Error" in payload:
            err = payload["Error"]
            code = err.get("code")
            msg = err.get("message", "")
            if msg == "NO_DATA_FOUND" or code == "50":
                return all_items
            raise RuntimeError(f"API error: {code} {msg}")

        parsed = PublicDataResponse[item_model].model_validate(payload)
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)

        if len(all_items) >= body.totalCount or not items:
            break
        page += 1
        time.sleep(0.1)

    return all_items


print("fetch 함수 준비 완료")

fetch 함수 준비 완료


In [4]:
# ============================================================
# 셀 4 - 파라미터 준비: 1개월(임시) × 서울 25구 + 테이블 + 재개
# ============================================================
# ⚠️ 현재 1개월(2025-10)만 테스트. 규모 확인 후 월 추가 여부 결정.
# 월 추가하려면 아래 리스트에 YYYYMM 추가만 하면 됨 (재개는 자동)

YEAR_MONTHS = ["202510"]   # 임시: 2025년 10월만

print(f"수집 대상 월: {YEAR_MONTHS}")

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS monthly_transfer_accessibility (
        opr_ym          TEXT,
        ctpv_cd         TEXT,
        sgg_cd          TEXT,
        emd_cd          TEXT,
        sttn_id         TEXT,
        inoutf_type_cd  TEXT,   -- I=유입, O=유출
        trnf_type_cd    TEXT,   -- TB/BT/BB/TT
        trfc_mns_se_cd  TEXT,   -- T=전철, B=버스
        tzon            TEXT,
        trnf_hr         INTEGER,  -- 총 환승시간(초)
        pasg_cnt        INTEGER,  -- 환승 건수
        pasg_nope       INTEGER,  -- 환승 이용자 수
        PRIMARY KEY (opr_ym, sttn_id, inoutf_type_cd, trnf_type_cd, tzon)
    )
    """)

with db_open(read_only=True) as con:
    sgg_list_df = con.execute("""
        SELECT sido_cd,
               sgg_cd,
               sido_cd || sgg_cd AS ctpv_sgg_5,
               locatadd_nm       AS sgg_nm
        FROM region
        WHERE sido_cd = '11'
          AND sgg_cd <> '000'
          AND umd_cd = '000'
          AND ri_cd = '00'
        ORDER BY sgg_cd
    """).df()

    done_df = con.execute("""
        SELECT DISTINCT opr_ym, sgg_cd FROM monthly_transfer_accessibility
    """).df()

done_set = set(zip(done_df["opr_ym"], done_df["sgg_cd"])) if len(done_df) > 0 else set()

all_combos = [
    (ym, r["ctpv_sgg_5"], r["sgg_nm"])
    for ym in YEAR_MONTHS
    for _, r in sgg_list_df.iterrows()
]
remaining = [(ym, sgg5, nm) for (ym, sgg5, nm) in all_combos if (ym, sgg5) not in done_set]

print(f"\n서울 시군구: {len(sgg_list_df)}개")
print(f"전체 조합: {len(all_combos)}개 ({len(YEAR_MONTHS)}월 × 25구)")
print(f"이미 완료: {len(all_combos) - len(remaining)}개 (재개 시 skip)")
print(f"남은 작업: {len(remaining)}개")

수집 대상 월: ['202510']

서울 시군구: 25개
전체 조합: 25개 (1월 × 25구)
이미 완료: 0개 (재개 시 skip)
남은 작업: 25개


In [5]:
# ============================================================
# 셀 5 - (월 × 서울 시군구) 순회, 시군구-월 단위 즉시 저장
# ============================================================
failed: list[tuple[str, str, str]] = []
started_at = datetime.now()
processed_total = 0

for i, (opr_ym, sgg_5, sgg_nm) in enumerate(remaining, start=1):
    try:
        items = fetch_all_pages(
            TRANSFER_URL,
            TransferAccessibilityItem,
            extra_params={
                "opr_ym": opr_ym,
                "ctpv_cd": "11",
                "sgg_cd":  sgg_5,
            },
        )

        if items:
            df_rows = pd.DataFrame([
                {
                    "opr_ym":          it.opr_ym,
                    "ctpv_cd":         it.ctpv_cd,
                    "sgg_cd":          it.sgg_cd,
                    "emd_cd":          it.emd_cd,
                    "sttn_id":         it.sttn_id,
                    "inoutf_type_cd":  it.inoutf_type_cd,
                    "trnf_type_cd":    it.trnf_type_cd,
                    "trfc_mns_se_cd":  it.trfc_mns_se_cd,
                    "tzon":            it.tzon,
                    "trnf_hr":         it.trnf_hr,
                    "pasg_cnt":        it.pasg_cnt,
                    "pasg_nope":       it.pasg_nope,
                }
                for it in items
            ])
            df_rows = df_rows.drop_duplicates(
                subset=["opr_ym","sttn_id","inoutf_type_cd","trnf_type_cd","tzon"]
            )
        else:
            df_rows = pd.DataFrame()

        if len(df_rows) > 0:
            with db_open() as con:
                con.register("df_view", df_rows)
                con.execute("""
                    INSERT INTO monthly_transfer_accessibility
                    SELECT opr_ym, ctpv_cd, sgg_cd, emd_cd,
                           sttn_id, inoutf_type_cd, trnf_type_cd, trfc_mns_se_cd,
                           tzon, trnf_hr, pasg_cnt, pasg_nope
                    FROM df_view
                    ON CONFLICT (opr_ym, sttn_id, inoutf_type_cd, trnf_type_cd, tzon)
                      DO NOTHING
                """)
                con.unregister("df_view")

        processed_total += len(df_rows)
        status = f"+{len(df_rows):>7,}건"
    except Exception as e:
        failed.append((opr_ym, sgg_5, str(e)[:100]))
        status = f"FAILED ({type(e).__name__})"

    elapsed = datetime.now() - started_at
    pct = i / len(remaining) * 100
    eta_sec = elapsed.total_seconds() / i * (len(remaining) - i)
    print(f"[{i:3d}/{len(remaining)}] {pct:5.1f}% | {opr_ym} {sgg_nm:15s} {status:20s} "
          f"| 누적 {processed_total:>10,} | 경과 {str(elapsed).split('.')[0]} | 잔여 ~{int(eta_sec//60)}분")

    time.sleep(0.1)

print(f"\n✅ 수집 완료")
print(f"   이번 실행 적재: {processed_total:,}건")
print(f"   실패 조합: {len(failed)}개")
print(f"   총 소요시간: {datetime.now() - started_at}")

with db_open(read_only=True) as con:
    grand_total = con.execute("SELECT COUNT(*) FROM monthly_transfer_accessibility").fetchone()[0]
print(f"\n📊 monthly_transfer_accessibility 총 누적: {grand_total:,}건")

[  1/25]   4.0% | 202510 서울특별시 종로구       + 24,514건            | 누적     24,514 | 경과 0:00:43 | 잔여 ~17분
[  2/25]   8.0% | 202510 서울특별시 중구        + 19,937건            | 누적     44,451 | 경과 0:02:06 | 잔여 ~24분
[  3/25]  12.0% | 202510 서울특별시 용산구       + 25,762건            | 누적     70,213 | 경과 0:03:35 | 잔여 ~26분
[  4/25]  16.0% | 202510 서울특별시 성동구       + 26,265건            | 누적     96,478 | 경과 0:04:49 | 잔여 ~25분
[  5/25]  20.0% | 202510 서울특별시 광진구       + 24,626건            | 누적    121,104 | 경과 0:05:41 | 잔여 ~22분
[  6/25]  24.0% | 202510 서울특별시 동대문구      + 26,031건            | 누적    147,135 | 경과 0:06:34 | 잔여 ~20분
[  7/25]  28.0% | 202510 서울특별시 중랑구       + 28,423건            | 누적    175,558 | 경과 0:08:00 | 잔여 ~20분
[  8/25]  32.0% | 202510 서울특별시 성북구       + 34,134건            | 누적    209,692 | 경과 0:09:25 | 잔여 ~20분
[  9/25]  36.0% | 202510 서울특별시 강북구       + 27,154건            | 누적    236,846 | 경과 0:10:30 | 잔여 ~18분
[ 10/25]  40.0% | 202510 서울특별시 도봉구       + 27,225건            | 누적    264,071 | 경과 0:11:36 

In [6]:
# ============================================================
# 셀 6 - (선택) 테이블 초기화
# ============================================================
RESET = False

if RESET:
    with db_open() as con:
        con.execute("DELETE FROM monthly_transfer_accessibility")
        cnt = con.execute("SELECT COUNT(*) FROM monthly_transfer_accessibility").fetchone()[0]
    print(f"⚠️  초기화 완료: {cnt}건")
else:
    print("RESET=False → 아무 작업 안 함")

RESET=False → 아무 작업 안 함


In [7]:
# ============================================================
# 셀 7 - 실패 조합 재시도
# ============================================================
if not failed:
    print("실패 없음 — 재시도 불필요")
else:
    print(f"재시도할 실패 조합: {len(failed)}개")
    retry_failed = []

    for opr_ym, sgg_5, prev_err in failed:
        try:
            items = fetch_all_pages(
                TRANSFER_URL,
                TransferAccessibilityItem,
                extra_params={"opr_ym": opr_ym, "ctpv_cd": "11", "sgg_cd": sgg_5},
            )
            if items:
                df_rows = pd.DataFrame([{
                    "opr_ym": it.opr_ym, "ctpv_cd": it.ctpv_cd,
                    "sgg_cd": it.sgg_cd, "emd_cd": it.emd_cd,
                    "sttn_id": it.sttn_id, "inoutf_type_cd": it.inoutf_type_cd,
                    "trnf_type_cd": it.trnf_type_cd, "trfc_mns_se_cd": it.trfc_mns_se_cd,
                    "tzon": it.tzon, "trnf_hr": it.trnf_hr,
                    "pasg_cnt": it.pasg_cnt, "pasg_nope": it.pasg_nope,
                } for it in items]).drop_duplicates(
                    subset=["opr_ym","sttn_id","inoutf_type_cd","trnf_type_cd","tzon"]
                )
                with db_open() as con:
                    con.register("df_view", df_rows)
                    con.execute("""
                        INSERT INTO monthly_transfer_accessibility
                        SELECT opr_ym, ctpv_cd, sgg_cd, emd_cd,
                               sttn_id, inoutf_type_cd, trnf_type_cd, trfc_mns_se_cd,
                               tzon, trnf_hr, pasg_cnt, pasg_nope
                        FROM df_view
                        ON CONFLICT (opr_ym, sttn_id, inoutf_type_cd, trnf_type_cd, tzon) DO NOTHING
                    """)
                    con.unregister("df_view")
                print(f"  ✅ {opr_ym}/{sgg_5}: +{len(df_rows)}건")
            else:
                print(f"  ⚠️  {opr_ym}/{sgg_5}: 빈 응답")
        except Exception as e:
            retry_failed.append((opr_ym, sgg_5, str(e)[:100]))
            print(f"  ❌ {opr_ym}/{sgg_5}: 다시 실패 — {e}")
        time.sleep(0.2)

    failed = retry_failed
    print(f"\n재시도 후 여전히 실패: {len(failed)}개")

실패 없음 — 재시도 불필요


In [8]:
# ============================================================
# 셀 8 - 검증 + 핵심 분석 쿼리
# ============================================================
with db_open(read_only=True) as con:
    print("=== 서울 월별 행수 ===")
    print(con.execute("""
        SELECT opr_ym, COUNT(*) AS rows,
               COUNT(DISTINCT sttn_id) AS unique_stops
        FROM monthly_transfer_accessibility
        GROUP BY opr_ym ORDER BY opr_ym
    """).df())

    print("\n=== 환승유형별 분포 (2025 전체) ===")
    print(con.execute("""
        SELECT trnf_type_cd, SUM(pasg_nope) AS 총승객, SUM(trnf_hr) AS 총환승시간초
        FROM monthly_transfer_accessibility
        GROUP BY trnf_type_cd
        ORDER BY 총승객 DESC
    """).df())

    # 핵심: 구별 평균 환승시간 (접근성 지표)
    print("\n=== 서울 구별 평균 환승시간 (초) - 접근성 지표 ===")
    print(con.execute("""
        SELECT mta.sgg_cd,
               ANY_VALUE(r.locallow_nm) AS sgg_nm,
               SUM(mta.pasg_cnt)                      AS 총환승건,
               ROUND(SUM(mta.trnf_hr)*1.0 / NULLIF(SUM(mta.pasg_cnt),0), 1) AS 평균환승초
        FROM monthly_transfer_accessibility mta
        LEFT JOIN region r
          ON r.sido_cd = mta.ctpv_cd
         AND r.sido_cd || r.sgg_cd = mta.sgg_cd
         AND r.umd_cd = '000' AND r.ri_cd = '00'
        GROUP BY mta.sgg_cd
        ORDER BY 평균환승초 DESC
    """).df())

    # 환승시간 긴 정류소 TOP 10 (노인 접근성 취약 예상)
    print("\n=== 평균 환승시간이 긴 TOP 10 정류소/역 ===")
    print(con.execute("""
        SELECT sttn_id,
               SUM(pasg_cnt) AS 총건수,
               ROUND(SUM(trnf_hr)*1.0/NULLIF(SUM(pasg_cnt),0),1) AS 평균환승초
        FROM monthly_transfer_accessibility
        WHERE trfc_mns_se_cd IS NOT NULL
        GROUP BY sttn_id
        HAVING SUM(pasg_cnt) >= 100
        ORDER BY 평균환승초 DESC
        LIMIT 10
    """).df())

=== 서울 월별 행수 ===
   opr_ym    rows  unique_stops
0  202510  825519         15240

=== 환승유형별 분포 (2025 전체) ===
  trnf_type_cd        총승객        총환승시간초
0           BB  7275881.0  4.703586e+09
1           BT  2129551.0  9.274670e+08
2           TB  1802236.0  1.295884e+09
3           TT   362303.0  1.152790e+08

=== 서울 구별 평균 환승시간 (초) - 접근성 지표 ===
   sgg_cd sgg_nm      총환승건  평균환승초
0   11140     중구  387444.0  657.3
1   11740    강동구  370402.0  640.5
2   11170    용산구  390565.0  639.3
3   11680    강남구  809422.0  638.2
4   11545    금천구  281052.0  636.3
5   11320    도봉구  311496.0  630.3
6   11650    서초구  707368.0  629.1
7   11500    강서구  527607.0  624.9
8   11110    종로구  390698.0  623.1
9   11440    마포구  559528.0  621.9
10  11620    관악구  495727.0  615.0
11  11200    성동구  329998.0  615.0
12  11215    광진구  357777.0  612.2
13  11380    은평구  497670.0  611.1
14  11560   영등포구  569586.0  607.5
15  11470    양천구  380811.0  604.7
16  11710    송파구  611146.0  599.6
17  11350    노원구  504819.0  599.6
18  11260